# 🧬 Orfipy ORF Prediction Examples

This notebook demonstrates ORF (Open Reading Frame) prediction using **Orfipy**.

- **Basic prediction** -- Predict ORFs in a single DNA sequence with default settings
- **Custom configuration** -- Tune start codons, minimum length, and strand filtering
- **Batch processing** -- Analyze multiple sequences at once with custom IDs
- **Result inspection** -- Examine individual ORF fields and metrics
- **Exporting** -- Save results as CSV or FASTA

## 📥 Imports

## 📦 Shared Data Types

### `ORF`
A single predicted open reading frame.

| Field | Type | Description |
|-------|------|-------------|
| `parent_id` | `str` | Identifier of the parent/input sequence |
| `orf_id` | `str` | Unique ORF identifier within the parent sequence |
| `strand` | `Literal["+", "-"]` | Strand direction |
| `frame` | `int` | Reading frame (`1`, `2`, or `3`) |
| `amino_acid_sequence` | `str` | Translated protein sequence |
| `nucleotide_sequence` | `str` | DNA sequence of the ORF |
| `amino_acid_length` | `int` | Length of protein in amino acids |
| `nucleotide_length` | `int` | Length of ORF in nucleotides |
| `nucleotide_start` | `int` | Start position (1-indexed, inclusive) |
| `nucleotide_end` | `int` | End position (1-indexed, inclusive) |
| `metrics` | `Dict[str, Any]` | Tool-specific metrics or metadata |
| `id` | `str` | Combined identifier: `parent_id` + `orf_id` (computed) |
| `gc_content` | `float` | GC content percentage (computed) |

In [1]:
from proto_tools import OrfipyConfig, OrfipyInput, run_orfipy_prediction


## 🔬 `run_orfipy_prediction`

Predict open reading frames in DNA sequences using Orfipy.

### API Reference

**Input: `OrfipyInput`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `List[str]` | *required* | DNA sequence(s) to analyze for open reading frames |
| `sequence_ids` | `Optional[List[str]]` | `None` | Optional sequence identifiers (defaults to `seq_0`, `seq_1`, ...) |

**Config: `OrfipyConfig`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `start_codons` | `list[StartCodon]` | `["ATG", "GTG", "TTG"]` | Start codons to recognize (multi-select from ATG, GTG, TTG, CTG) |
| `stop_codons` | `list[StopCodon]` | `["TAA", "TAG", "TGA"]` | Stop codons to recognize (multi-select from TAA, TAG, TGA) |
| `strand` | `Literal["f", "r", "b"]` | `"b"` | Which strand(s) to scan: `"f"` (forward), `"r"` (reverse), or `"b"` (both) |
| `min_len` | `int` | `0` | Minimum ORF length in nucleotides |
| `max_len` | `int` | `10000` | Maximum ORF length in nucleotides |
| `include_stop` | `bool` | `True` | Whether to include the stop codon in the reported ORF |
| `translation_table` | `OrfipyTranslationTable \| None` | `None` | NCBI genetic code name (None = standard genetic code) |

**Output: `OrfipyOutput`**

| Field | Type | Description |
|-------|------|-------------|
| `predicted_orfs` | `List[List[ORF]]` | List of ORF results per input sequence |
| `num_orfs` | `int` | Total number of ORFs predicted (computed) |
| `num_orfs_per_sequence` | `List[int]` | Number of ORFs per input sequence (computed) |

---

### 1. Basic Single-Sequence ORF Prediction

Predict ORFs in a short DNA sequence using default settings.

In [ ]:
import pandas as pd

# A short DNA sequence with a known ORF
sequence = "ATGGTGCTGAGCCCGGCGGACAAGACCAACGTGAAGGCGGCGTGGGGCAAGTGA"

inputs = OrfipyInput(sequences=sequence)
config = OrfipyConfig(min_len=30)

result = run_orfipy_prediction(inputs, config)
print(f"Found {result.num_orfs} ORFs")
pd.DataFrame([orf.to_flat_dict() for orfs in result.predicted_orfs for orf in orfs])

### 2. Custom Configuration

Customize start codons, minimum length, and strand filtering.

In [ ]:
# Only ATG start codon, forward strand only, minimum 12 nt
config_custom = OrfipyConfig(
    start_codons=["ATG"],
    min_len=12,
    strand="f",
    include_stop=True,
)

result_custom = run_orfipy_prediction(inputs, config_custom)
print(f"Found {result_custom.num_orfs} ORFs (forward strand, ATG only)")
pd.DataFrame([orf.to_flat_dict() for orfs in result_custom.predicted_orfs for orf in orfs])

### 3. Batch Processing with Multiple Sequences

Process multiple sequences at once with custom IDs.

In [ ]:
sequences = [
    "ATGCCCAAATTTGGGCCCAAATTTGGGCCCAAATTTGGGTAG",
    "ATGTTTCCCGGGAAATTTCCCGGGTAA",
    "ATGAAACCCGGGAAATTTCCCGGGAAATTTCCCGGGAAATTTCCCGGGTAG",
]

inputs_batch = OrfipyInput(
    sequences=sequences,
    sequence_ids=["gene_alpha", "gene_beta", "gene_gamma"],
)
config_batch = OrfipyConfig(min_len=12)

result_batch = run_orfipy_prediction(inputs_batch, config_batch)
print(f"Total ORFs: {result_batch.num_orfs}")
print(f"ORFs per sequence: {result_batch.num_orfs_per_sequence}")
pd.DataFrame([orf.to_flat_dict() for orfs in result_batch.predicted_orfs for orf in orfs])

### 4. Examining Individual ORF Results

In [5]:
if result_batch.num_orfs > 0:
    orf = result_batch.predicted_orfs[0][0]
    print(f"Parent ID: {orf.parent_id}")
    print(f"ORF ID: {orf.orf_id}")
    print(f"Strand: {orf.strand}")
    print(f"Frame: {orf.frame}")
    print(f"Position: {orf.nucleotide_start}-{orf.nucleotide_end}")
    print(f"AA length: {orf.amino_acid_length}")
    print(f"NT length: {orf.nucleotide_length}")
    print(f"Protein: {orf.amino_acid_sequence}")

Parent ID: gene_alpha
ORF ID: ORF.1
Strand: +
Frame: 1
Position: 1-42
AA length: 13
NT length: 42
Protein: MPKFGPKFGPKFG


## 💾 Export Results

Save results to files for downstream analysis.

- **CSV** -- Tabular format with all ORF fields
- **FAA** -- Protein sequences in FASTA format

In [6]:
# Export to CSV
result_batch.export("orfipy_results", export_path="./example_output", file_format="csv")
print("Exported to ./example_output/orfipy_results.csv")

# Export protein sequences as FASTA
result_batch.export("orfipy_results", export_path="./example_output", file_format="faa")
print("Exported to ./example_output/orfipy_results.faa")

Exported to ./example_output/orfipy_results.csv
Exported to ./example_output/orfipy_results.faa
